In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.metrics import mean_squared_error, root_mean_squared_error

In [2]:
import pickle
import mlflow


In [3]:


mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("nyc-taxi-experiment")

<Experiment: artifact_location='/workspaces/mlops/experiment-tracking/mlruns/1', creation_time=1745253509108, experiment_id='1', last_update_time=1745253509108, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}>

In [4]:
def read_frame(file_name="../data/green_tripdata_2021-01.parquet"):
    df = pd.read_parquet(file_name)

    duration =  df.lpep_dropoff_datetime - df.lpep_pickup_datetime 
    df.loc[:,"duration"] = duration

    df.duration = df.duration.apply(lambda td: td.total_seconds()/60)
    df  = df[((df.duration >=1) & (df.duration<=60))]

    categorical_variables = ['PULocationID', 'DOLocationID']

    df[categorical_variables] = df[categorical_variables].astype(str)
    
    return df

In [5]:
df_train = read_frame()
df_val = read_frame(file_name="../data/green_tripdata_2021-02.parquet")


In [6]:
len(df_train), len(df_val)

(73908, 61921)

In [7]:
df_train["PU_DO"] = df_train['PULocationID']+"_"+ df_train['DOLocationID']
df_val["PU_DO"] = df_val['PULocationID']+"_"+ df_val['DOLocationID']

In [8]:
numerical = ["trip_distance"]
categorical_variables = ["PU_DO"]#['PULocationID', 'DOLocationID']

train_dict = df_train[categorical_variables+numerical].to_dict(orient="records")
dv = DictVectorizer()
X_train = dv.fit_transform(train_dict)

val_dict = df_val[categorical_variables+numerical].to_dict(orient="records")
X_val = dv.transform(val_dict)



In [9]:
target = "duration"
y_train = df_train[target].values
y_val = df_val[target].values

In [10]:
lr = LinearRegression()

lr.fit(X_train, y_train)

y_pred = lr.predict(X_val)


mean_squared_error(y_val, y_pred)

60.19766170466965

In [11]:
with mlflow.start_run():
    
    mlflow.set_tag("developer", "Tousside")
    
    mlflow.log_param("train-data-path", "../data/green_tripdata_2021-01.parquet")
    mlflow.log_param("val-data-path", "../data/green_tripdata_2021-02.parquet")
    
    alpha = 0.1
    mlflow.log_param("alpha", alpha)
    mlflow.log_param("model-name", "lasso-regression")
    
    lr = Lasso(alpha=alpha)

    lr.fit(X_train, y_train)

    y_pred = lr.predict(X_val)


    rmse = root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric("rmse", rmse)
    
    mlflow.log_artifact(local_path = "models/lasso_reg.bin", artifact_path="models_pickle")

In [12]:
# lr = Lasso(alpha=0.0001)

# lr.fit(X_train, y_train)

# y_pred = lr.predict(X_val)


# mean_squared_error(y_val, y_pred)

In [13]:
# with open("../models/lasso_reg.bin", "wb") as f:
#     pickle.dump((dv, lr), f)

In [14]:
# lr = Ridge(alpha=2)

# lr.fit(X_train, y_train)

# y_pred = lr.predict(X_val)


# mean_squared_error(y_val, y_pred)

In [15]:
import xgboost as xgb

from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll import scope

In [16]:
train = xgb.DMatrix(X_train, label=y_train)
valid = xgb.DMatrix(X_val, label=y_val)

# Big search

In [17]:
def objective(params):
    with mlflow.start_run():
        mlflow.set_tag("model", "xgboost")
        mlflow.log_params(params)
        booster = xgb.train(
            params = params,
            dtrain = train,
            num_boost_round = 1000,
            evals = [(valid, "validation")],
            early_stopping_rounds =50 
        )
        
        ypred = booster.predict(valid)
        
        rmse = root_mean_squared_error(y_val, ypred)
        
        mlflow.log_metric("rmse", rmse)
        
        return {"loss": rmse, "status": STATUS_OK}
        

In [ ]:
search_space = {
    "max_depth": scope.int(hp.quniform("max_depth", 4, 100, 1)),
    "learning_rate": hp.loguniform("learning_rate", -3, 0),
    "reg_alpha":  hp.loguniform("reg_alpha", -5, -1),
    "reg_lambda":  hp.loguniform("reg_lambda", -6, -1),
    "min_child_weight":  hp.loguniform("min_child_weight", -1, 3),
    "objective":  "reg:linear",
    "seed": 42
}

best_result = fmin(
    fn = objective,
    space = search_space,
    algo=tpe.suggest, 
    max_evals = 50,
    trials = Trials()
)

# Saving the models directely

In [ ]:
# mlflow.xgboost.autolog(disable=False)

In [19]:
best_params = {
    "learning_rate": 0.15305107685333966,
    "max_depth" :28, 
    "min_child_weight" :1.5718950485784093,
    "objective": "reg:squarederror",
    "reg_alpha": 0.043377919442635034,
    "reg_lambda": 0.04957668195902974,
    "seed": 42
     
 }

mlflow.log_params(best_params)
 
booster = xgb.train(
        params = best_params,
        dtrain = train,
        num_boost_round = 1000,
        evals = [(valid, "validation")],
        early_stopping_rounds =50 
    )

y_pred = booster.predict(valid)
rmse = root_mean_squared_error(y_val, y_pred)
mlflow.log_metric("rmse", rmse)

with open("models/preprocessor.b", "wb") as f:
    pickle.dump(dv, f)
    
mlflow.log_artifact("models/preprocessor.b", artifact_path="preprocesor")

mlflow.xgboost.log_model(booster, artifact_path="models_mlflow")

[0]	validation-rmse:10.99981
[1]	validation-rmse:10.02874
[2]	validation-rmse:9.25675
[3]	validation-rmse:8.65211
[4]	validation-rmse:8.18244
[5]	validation-rmse:7.81567
[6]	validation-rmse:7.53552
[7]	validation-rmse:7.31883
[8]	validation-rmse:7.15188
[9]	validation-rmse:7.02266
[10]	validation-rmse:6.92388
[11]	validation-rmse:6.84822
[12]	validation-rmse:6.78712
[13]	validation-rmse:6.73893
[14]	validation-rmse:6.70013
[15]	validation-rmse:6.66838
[16]	validation-rmse:6.64284
[17]	validation-rmse:6.62231
[18]	validation-rmse:6.60551
[19]	validation-rmse:6.59291
[20]	validation-rmse:6.58037
[21]	validation-rmse:6.57153
[22]	validation-rmse:6.56238
[23]	validation-rmse:6.55523
[24]	validation-rmse:6.54916
[25]	validation-rmse:6.54460
[26]	validation-rmse:6.53978
[27]	validation-rmse:6.53678
[28]	validation-rmse:6.53414
[29]	validation-rmse:6.53255
[30]	validation-rmse:6.53128
[31]	validation-rmse:6.53092
[32]	validation-rmse:6.52881
[33]	validation-rmse:6.52698
[34]	validation-rmse:6

/workspaces/mlops/venv/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [21:10:07] WARNING: /workspace/src/c_api/c_api.cc:1374: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  warnings.warn(smsg, UserWarning)
2025/04/21 21:10:11 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


In [ ]:
params = {
    "learning_rate": 0.15305107685333966,
    "max_depth" :28, 
    "min_child_weight" :1.5718950485784093,
    "objective": "reg:squarederror",
    "reg_alpha": 0.043377919442635034,
    "reg_lambda": 0.04957668195902974,
    "seed": 42
     
 }

mlflow.xgboost.autolog()
 
booster = xgb.train(
        params = params,
        dtrain = train,
        num_boost_round = 1000,
        evals = [(valid, "validation")],
        early_stopping_rounds =50 
    )

2025/04/21 18:31:45 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '58401724c53e4ed186eac436aa731d76', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current xgboost workflow


# prediction

In [20]:
logged_model = 'runs:/2668fd05b17d4754bcb457a80c02897a/models_mlflow'

# Load model as a PyFuncModel.
loaded_model = mlflow.pyfunc.load_model(logged_model)



In [21]:
loaded_model

mlflow.pyfunc.loaded_model:
  artifact_path: models_mlflow
  flavor: mlflow.xgboost
  run_id: 2668fd05b17d4754bcb457a80c02897a

In [22]:
xgboost_model = mlflow.xgboost.load_model(logged_model)

In [23]:
ypred = xgboost_model.predict(valid)

In [24]:
y_pred[:10]

array([14.30691  ,  7.2030396, 15.272669 , 24.452576 ,  9.426713 ,
       17.138124 , 11.007457 ,  7.9626713,  9.445903 , 19.148085 ],
      dtype=float32)

In [ ]:


fig, ax = plt.subplots()
sns.histplot(y_pred, label='Prediction', ax=ax, kde=True)
sns.histplot(y_train, label='Actual', ax=ax, kde=True)

ax.legend()
plt.show()


In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_train)